# Bruit Suisse + Auvergne-Rhône-Alpes en classes ORHANE (1 à 7) — Grand Genève

Ce notebook :

1. lit la couche **suisse en dB(A) (Lr)** et la reclasse selon les seuils de
   l'**indicateur Bruit ORHANE** (classes 1 à 7, pas de 5 dB) ;
2. lit la couche **ORHANE** (`sous_indice_multibruit_aura.tif`), déjà en
   classes 1 à 7 (-99 = hors région, y compris la Suisse et le lac) ;
3. reprojette tout sur une **grille commune en CH1903+/LV95 (EPSG:2056)** ;
4. fait l'**union** des deux couches et la **découpe au périmètre du Grand
   Genève** ;
5. **normalise les classes entre 0 et 1** : `classe_norm = (classe − 1) / 6`,
   donc 1 → 0, 4 → 0,5, 7 → 1 ;
6. exporte en **GeoPackage, CSV et Parquet** (un point par pixel), plus un
   GeoTIFF en classes avec la palette ORHANE pour QGIS.

| Classe | Zone | Lden |
|---|---|---|
| 1 | Préservée ou absence de donnée | < 50 |
| 2 | Peu altérée | 50 – 55 |
| 3 | Moyennement altérée | 55 – 60 |
| 4 | Altérée | 60 – 65 |
| 5 | Dégradée | 65 – 70 |
| 6 | Très dégradée | 70 – 75 |
| 7 | Hautement dégradée | ≥ 75 |

> ⚠️ Côté suisse, les seuils Lden sont appliqués à un Lr : c'est une
> approximation (écart typique de quelques dB, inférieur à la largeur d'une
> classe). Autre asymétrie : en Suisse, la classe 1 veut vraiment dire « moins
> de 50 dB », alors que côté ORHANE elle mélange « moins de 50 dB » et
> « pas de donnée » — d'où l'option `FR_CLASSE_1` ci-dessous.

## Dépendances

In [1]:
import importlib.util
import subprocess
import sys

REQUIRED = ["rasterio", "geopandas", "shapely", "pyproj", "pyarrow", "numpy", "pandas", "pyogrio"]
missing = [pkg for pkg in REQUIRED if importlib.util.find_spec(pkg) is None]
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", *missing])

from pathlib import Path

import geopandas as gpd
import numpy as np
import pandas as pd
import rasterio
from rasterio.features import geometry_mask
from rasterio.transform import from_origin
from rasterio.warp import Resampling, reproject, transform_bounds
from rasterio.windows import Window, from_bounds

print("rasterio", rasterio.__version__, "| geopandas", gpd.__version__, "| numpy", np.__version__)

rasterio 1.4.3 | geopandas 1.0.1 | numpy 2.0.0


## Paramètres

- `FR_CLASSE_1` : que faire des pixels ORHANE en classe 1 (« < 50 dB ou
  absence de donnée ») :
  - `"45dB"` : on les garde en **classe 1**, avec 45 dB comme valeur
    indicative dans la colonne `bruit_db_approx` ;
  - `"nan"` : on les considère comme **sans donnée** (exclus de l'export).
- `RESOLUTION_M` : taille de pixel de la grille commune. Les deux sources sont
  à 10 m, mais à 10 m le Grand Genève donne ~18 millions de points (GPKG de
  plusieurs Go). 25 m est un bon compromis ; mets `None` pour garder la
  résolution native la plus grossière des deux.
- `OVERLAP_RULE` : règle là où les deux couches ont une valeur (en pratique
  quelques pixels le long de la frontière, car ORHANE est à -99 en Suisse) :
  `"max"`, `"priorite_ch"` ou `"priorite_fr"`.
- `CH_EXTRA_NODATA` : valeurs du raster suisse à traiter comme « pas de
  donnée » en plus du nodata déclaré (ex. `[0]`).

In [2]:
CH_TIF = "/Users/maximiliantrique/Downloads/laerm-strassenlaerm_tag_2056.tif"                     # raster suisse en dB(A), Lr
FR_TIF = "/Users/maximiliantrique/Downloads/orhane_2023_multibruit/sous_indice_multibruit_aura.tif"      # ORHANE, classes 1..7, nodata -99

PERIMETER_VARIANTS = {
    "avec_lac": {"kind": "shp", "path": "/Users/maximiliantrique/Documents/GitHub/projet_pieton_gg/Data/input/network_agreg/AGGLO_PERIMETRE_AVEC_LAC-SHP/AGGLO_PERIMETRE_AVEC_LAC.shp"},
    "sans_lac": {"kind": "parquet", "path": "/Users/maximiliantrique/Documents/GitHub/projet_pieton_gg/Data/input/network_agreg/AGGLO_PERIMETRE_SANS_LAC-parquet/AGGLO_PERIMETRE_SANS_LAC.parquet"},
}
PERIMETER_VARIANT = "avec_lac"

FR_CLASSE_1 = "45dB"            # "45dB" | "nan"
OVERLAP_RULE = "max"            # "max" | "priorite_ch" | "priorite_fr"
RESOLUTION_M = 10               # None = résolution native la plus grossière
CH_EXTRA_NODATA = []            # ex. [0]

TARGET_CRS = "EPSG:2056"        # CH1903+ / LV95
CLASS_BREAKS_DB = [50, 55, 60, 65, 70, 75]   # seuils ORHANE (Lden), bornes basses incluses
# valeur indicative en dB par classe (milieu de classe ; 7 est ouverte vers le haut)
CLASS_TO_DB = {1: 45.0, 2: 52.5, 3: 57.5, 4: 62.5, 5: 67.5, 6: 72.5, 7: 77.5}

RESAMPLING_CH = Resampling.average   # dB continus : moyenne si on agrège des pixels
RESAMPLING_FR = Resampling.mode      # classes : valeur majoritaire (jamais d'interpolation)

ADD_LONLAT = True
OUT_DIR = Path("sorties_bruit")
OUT_STEM = "bruit_classes_grand_geneve"

## Lecture des rasters et du périmètre

In [3]:
def describe(path):
    with rasterio.open(path) as src:
        if src.crs is None:
            raise ValueError(f"{path} n'a pas de SCR déclaré : impossible de le reprojeter.")
        print(f"{path}\n  SCR    : {src.crs.to_string()}\n  taille : {src.width} x {src.height}"
              f"\n  pixel  : {src.res[0]:g} x {src.res[1]:g}\n  nodata : {src.nodata}"
              f"\n  type   : {src.dtypes[0]} ({src.count} bande(s), seule la 1re est lue)")
        return src.crs, src.res

ch_crs, ch_res = describe(CH_TIF)
fr_crs, fr_res = describe(FR_TIF)


def load_perimeter(variant):
    spec = PERIMETER_VARIANTS[variant]
    gdf = gpd.read_file(spec["path"]) if spec["kind"] == "shp" else gpd.read_parquet(spec["path"])
    return gdf.to_crs(TARGET_CRS).union_all()

perimeter = load_perimeter(PERIMETER_VARIANT)
print(f"\nPérimètre « {PERIMETER_VARIANT} » : {perimeter.area / 1e6:,.0f} km²")

/Users/maximiliantrique/Downloads/laerm-strassenlaerm_tag_2056.tif
  SCR    : EPSG:2056
  taille : 38202 x 24449
  pixel  : 10 x 10
  nodata : -9999.0
  type   : float64 (1 bande(s), seule la 1re est lue)
/Users/maximiliantrique/Downloads/orhane_2023_multibruit/sous_indice_multibruit_aura.tif
  SCR    : EPSG:2154
  taille : 40119 x 29549
  pixel  : 10 x 10
  nodata : -99.0
  type   : int16 (1 bande(s), seule la 1re est lue)

Périmètre « avec_lac » : 2,213 km²


## Grille commune en EPSG:2056

In [4]:
def res_in_metres(crs, res):
    if crs.is_geographic:
        return max(res) * 111_000 * np.cos(np.radians(46.2))
    return max(res) * crs.linear_units_factor[1]

res = float(RESOLUTION_M or max(res_in_metres(ch_crs, ch_res), res_in_metres(fr_crs, fr_res)))

xmin, ymin, xmax, ymax = perimeter.bounds
xmin, ymin = np.floor(xmin / res) * res, np.floor(ymin / res) * res
xmax, ymax = np.ceil(xmax / res) * res, np.ceil(ymax / res) * res
width, height = int(round((xmax - xmin) / res)), int(round((ymax - ymin) / res))
dst_transform = from_origin(xmin, ymax, res, res)

inside = ~geometry_mask([perimeter], out_shape=(height, width), transform=dst_transform)
print(f"Grille : {width} x {height} pixels de {res:g} m — {inside.sum():,} pixels dans le périmètre")

Grille : 6555 x 6893 pixels de 10 m — 22,127,419 pixels dans le périmètre


## Reprojection

On ne lit que la partie de chaque raster qui couvre le Grand Genève. Côté
ORHANE, la règle `FR_CLASSE_1` est appliquée **après** la reprojection, pour que
la classe majoritaire soit calculée sur les vraies classes 1 à 7.

In [5]:
def read_window_and_reproject(path, prepare, resampling):
    with rasterio.open(path) as src:
        margin = 5 * res
        b = transform_bounds(TARGET_CRS, src.crs, xmin - margin, ymin - margin, xmax + margin, ymax + margin)
        win = from_bounds(*b, transform=src.transform).round_offsets().round_lengths()
        win = win.intersection(Window(0, 0, src.width, src.height))
        a = prepare(src.read(1, window=win).astype("float32"), src.nodata)
        out = np.full((height, width), np.nan, dtype="float32")
        reproject(a, out,
                  src_transform=src.window_transform(win), src_crs=src.crs, src_nodata=np.nan,
                  dst_transform=dst_transform, dst_crs=TARGET_CRS, dst_nodata=np.nan,
                  resampling=resampling)
    return out


def prepare_ch(a, nodata):
    bad = ~np.isfinite(a)
    if nodata is not None and not np.isnan(nodata):
        bad |= a == nodata
    for v in CH_EXTRA_NODATA:
        bad |= a == v
    a[bad] = np.nan
    return a


def prepare_fr(a, nodata):
    a[~np.isin(a, np.arange(1, 8))] = np.nan          # -99 et toute valeur hors 1..7
    return a


ch_db = read_window_and_reproject(CH_TIF, prepare_ch, RESAMPLING_CH)
fr_cls = read_window_and_reproject(FR_TIF, prepare_fr, RESAMPLING_FR)

# Appliqué APRÈS la reprojection : sinon la classe majoritaire serait calculée
# sans la classe 1 et gonflerait artificiellement les classes bruyantes.
if FR_CLASSE_1 == "nan":
    fr_cls[fr_cls == 1] = np.nan
elif FR_CLASSE_1 != "45dB":
    raise ValueError("FR_CLASSE_1 doit valoir '45dB' ou 'nan'")

## Reclassement de la couche suisse

`np.digitize` avec les seuils `[50, 55, 60, 65, 70, 75]` donne 0 pour < 50,
1 pour 50–55, …, 6 pour ≥ 75 ; on ajoute 1 pour obtenir les classes 1 à 7
(bornes basses incluses, comme dans le tableau ORHANE : 50 ≤ Lden < 55 → 2).

In [6]:
ch_cls = np.where(np.isfinite(ch_db), np.digitize(ch_db, CLASS_BREAKS_DB) + 1, np.nan).astype("float32")

for name, arr in [("Suisse", ch_cls), ("ORHANE", fr_cls)]:
    v = arr[inside & np.isfinite(arr)].astype(int)
    counts = pd.Series(v).value_counts().sort_index()
    print(f"{name} — pixels par classe dans le périmètre :\n{counts.to_string()}\n")

Suisse — pixels par classe dans le périmètre :
1    3027518
2     574642
3     400542
4     233658
5     111961
6      41339
7      26973

ORHANE — pixels par classe dans le périmètre :
1    11274396
2     1177021
3      805632
4      618574
5      330011
6      147121
7       92592



## Union, découpe et normalisation 0–1

In [7]:
has_ch, has_fr = np.isfinite(ch_cls), np.isfinite(fr_cls)
both = has_ch & has_fr

classe = np.where(has_ch, ch_cls, fr_cls)
if OVERLAP_RULE == "max":
    classe[both] = np.maximum(ch_cls[both], fr_cls[both])
elif OVERLAP_RULE == "priorite_ch":
    classe[both] = ch_cls[both]
elif OVERLAP_RULE == "priorite_fr":
    classe[both] = fr_cls[both]
else:
    raise ValueError(f"OVERLAP_RULE inconnue : {OVERLAP_RULE}")
classe[~inside] = np.nan

classe_norm = (classe - 1) / 6                       # 1 -> 0, 7 -> 1

# Valeur indicative en dB : Lr suisse réel côté CH, milieu de classe côté FR
lut = np.full(8, np.nan, dtype="float32")
for k, v in CLASS_TO_DB.items():
    lut[k] = v
fr_db = np.where(has_fr, lut[np.nan_to_num(fr_cls, nan=0).astype(int)], np.nan)
choix_ch = has_ch & (~both | (classe == ch_cls))
bruit_db = np.where(choix_ch, ch_db, fr_db)

source = np.select([both, has_ch, has_fr], ["CH+FR", "CH", "FR"], default="")

print(f"Pixels avec une classe : {np.isfinite(classe).sum():,} "
      f"(dont {(both & inside).sum():,} en recouvrement, règle « {OVERLAP_RULE} »)")
print(f"Pixels du périmètre sans donnée (lac, hors couverture, classe 1 FR si 'nan') : "
      f"{(inside & ~np.isfinite(classe)).sum():,}")

Pixels avec une classe : 18,686,850 (dont 175,130 en recouvrement, règle « max »)
Pixels du périmètre sans donnée (lac, hors couverture, classe 1 FR si 'nan') : 3,440,569


## Export

Colonnes : `classe` (1–7), `classe_norm` (0–1), `bruit_db_approx` (Lr suisse
réel ou valeur indicative ORHANE), `classe_ch`, `classe_fr` (chaque source
seule, vide si absente), `source` (`CH`, `FR`, `CH+FR`), coordonnées LV95
(+ `lon`, `lat`).

Le GeoTIFF contient les classes en entiers (0 = pas de donnée) avec la palette
ORHANE intégrée : QGIS l'affiche directement avec les bonnes couleurs.

In [8]:
OUT_DIR.mkdir(parents=True, exist_ok=True)

PALETTE = {1: "#6ee86e", 2: "#c9ff8f", 3: "#ffff3c", 4: "#f5c04a",
           5: "#ec3a2a", 6: "#c63cf5", 7: "#6f2f9f"}
hex2rgba = lambda h: tuple(int(h[i:i + 2], 16) for i in (1, 3, 5)) + (255,)

with rasterio.open(OUT_DIR / f"{OUT_STEM}.tif", "w", driver="GTiff", width=width, height=height,
                   count=1, dtype="uint8", crs=TARGET_CRS, transform=dst_transform,
                   nodata=0, compress="deflate") as dst:
    dst.write(np.nan_to_num(classe, nan=0).astype("uint8"), 1)
    dst.write_colormap(1, {0: (0, 0, 0, 0), **{k: hex2rgba(v) for k, v in PALETTE.items()}})

rows, cols = np.nonzero(np.isfinite(classe))
xs = xmin + (cols + 0.5) * res
ys = ymax - (rows + 0.5) * res

df = pd.DataFrame({
    "x_lv95": xs,
    "y_lv95": ys,
    "classe": classe[rows, cols].astype("int8"),
    "classe_norm": classe_norm[rows, cols].round(4),
    "bruit_db_approx": bruit_db[rows, cols].round(1),
    "classe_ch": pd.array(ch_cls[rows, cols], dtype="Float32").astype("Int8"),
    "classe_fr": pd.array(fr_cls[rows, cols], dtype="Float32").astype("Int8"),
    "source": source[rows, cols],
})
gdf = gpd.GeoDataFrame(df, geometry=gpd.points_from_xy(df.x_lv95, df.y_lv95), crs=TARGET_CRS)
if ADD_LONLAT:
    wgs = gdf.geometry.to_crs(4326)
    gdf["lon"], gdf["lat"] = wgs.x.round(6).to_numpy(), wgs.y.round(6).to_numpy()

gdf.to_file(OUT_DIR / f"{OUT_STEM}.gpkg", layer="bruit_classes", driver="GPKG", engine="pyogrio")
gdf.to_parquet(OUT_DIR / f"{OUT_STEM}.parquet")
gdf.drop(columns="geometry").to_csv(OUT_DIR / f"{OUT_STEM}.csv", index=False)

for f in sorted(OUT_DIR.glob(f"{OUT_STEM}.*")):
    print(f"{f}  ({f.stat().st_size / 1e6:,.1f} Mo)")
gdf.head()

sorties_bruit/bruit_classes_grand_geneve.csv  (1,053.7 Mo)
sorties_bruit/bruit_classes_grand_geneve.gpkg  (2,586.3 Mo)
sorties_bruit/bruit_classes_grand_geneve.parquet  (325.8 Mo)
sorties_bruit/bruit_classes_grand_geneve.tif  (1.5 Mo)


,x_lv95,y_lv95,classe,classe_norm,bruit_db_approx,classe_ch,classe_fr,source,geometry,lon,lat
0,2501665.0,1155685.0,1,0.0,44.599998,1,<NA>,CH,POINT (2501665 1155685),6.156534,46.545208
1,2501675.0,1155685.0,1,0.0,44.400002,1,<NA>,CH,POINT (2501675 1155685),6.156664,46.545209
2,2501655.0,1155675.0,1,0.0,44.400002,1,<NA>,CH,POINT (2501655 1155675),6.156405,46.545116
3,2501665.0,1155675.0,1,0.0,44.099998,1,<NA>,CH,POINT (2501665 1155675),6.156536,46.545118
4,2501675.0,1155675.0,1,0.0,43.900002,1,<NA>,CH,POINT (2501675 1155675),6.156666,46.545119


## Contrôle rapide

In [9]:
print(pd.crosstab(gdf["classe"], gdf["source"], margins=True))
print()
print(gdf.groupby("classe")["classe_norm"].first())

source       CH   CH+FR        FR       All
classe                                     
1       2857581   83727  11188509  14129817
2        570610   21630   1156889   1749129
3        399783   16091    789927   1205801
4        233283   16441    602395    852119
5        111935   16590    313434    441959
6         41338    7982    139140    188460
7         26973   12669     79923    119565
All     4241503  175130  14270217  18686850

classe
1    0.0000
2    0.1667
3    0.3333
4    0.5000
5    0.6667
6    0.8333
7    1.0000
Name: classe_norm, dtype: float32


In [10]:
# === Export agrégé au carroyage 200 m (SITG, AGGLO_CARREAU_200) ===
# Chaque pixel de la grille commune est rattaché au carreau qui contient son centre
# (à 10 m : exactement 20 x 20 = 400 pixels par carreau, aucun pixel n'est coupé).
# Source carroyage : « Système d'information du territoire à Genève (SITG), extrait en
# date du 31.03.2025 » — agrégation du bruit réalisée sur cette base (CGU SITG, art. 9).

CARREAU_PATH = "/Users/maximiliantrique/Documents/GitHub/projet_pieton_gg/Data/input/network_agreg/AGGLO_CARREAU_200-SHP/AGGLO_CARREAU_200.shp"  # à adapter
CARREAU_ID = "GRID_ID"
CARREAU_MIN_COUVERTURE = 0.0     # part mini de pixels avec donnée pour garder un carreau (0 = au moins 1 pixel)
CARREAU_STEM = f"{OUT_STEM}_carreau200"

carreaux = gpd.read_file(CARREAU_PATH).to_crs(TARGET_CRS).reset_index(drop=True)

# --- géométrie de la grille, déduite des carreaux eux-mêmes ---
bds = carreaux.geometry.bounds
pas = float(np.median(bds.maxx - bds.minx))
x0, y0 = float(bds.minx.min()), float(bds.miny.min())
ci = np.round((bds.minx - x0) / pas).astype(int).to_numpy()
cj = np.round((bds.miny - y0) / pas).astype(int).to_numpy()
assert np.allclose(bds.minx - x0, ci * pas, atol=1) and np.allclose(bds.miny - y0, cj * pas, atol=1), \
    "Carroyage non régulier : l'indexation arithmétique ne s'applique pas."
ncol, nrow = ci.max() + 1, cj.max() + 1
cell_lut = np.full(nrow * ncol, -1, dtype=np.int64)
cell_lut[cj * ncol + ci] = np.arange(len(carreaux))

if abs(pas / res - round(pas / res)) > 1e-6:
    print(f"⚠️ {pas:g} m n'est pas un multiple de la résolution ({res:g} m) : "
          "le nombre de pixels par carreau varie, la couverture est approximative.")
n_theo = (pas / res) ** 2

# --- rattachement pixel -> carreau (pixels du périmètre seulement) ---
r, c = np.nonzero(inside)
px = xmin + (c + 0.5) * res
py = ymax - (r + 0.5) * res
pi = np.floor((px - x0) / pas).astype(np.int64)
pj = np.floor((py - y0) / pas).astype(np.int64)
dans_grille = (pi >= 0) & (pi < ncol) & (pj >= 0) & (pj < nrow)
idx = np.full(r.shape, -1, dtype=np.int64)
idx[dans_grille] = cell_lut[pj[dans_grille] * ncol + pi[dans_grille]]
garde = idx >= 0
r, c, idx = r[garde], c[garde], idx[garde]
print(f"{pas:g} m, {len(carreaux):,} carreaux — {garde.sum():,} pixels du périmètre rattachés "
      f"({(~garde).sum():,} hors carroyage)")

n = len(carreaux)
cnt = lambda m, w=None: np.bincount(idx[m], weights=None if w is None else w[m], minlength=n)

cls_px = classe[r, c]
ok = np.isfinite(cls_px)
n_perim = np.bincount(idx, minlength=n)
n_data = cnt(ok)

# effectifs par classe -> parts, classe majoritaire, classe max
par_classe = np.stack([cnt(ok & (cls_px == k)) for k in range(1, 8)], axis=1)   # (n, 7)
with np.errstate(invalid="ignore", divide="ignore"):
    parts = par_classe / n_data[:, None]
    classe_norm_moy = cnt(ok, classe_norm[r, c]) / n_data
    classe_moy = cnt(ok, cls_px) / n_data
    # moyenne énergétique en dB : 10·log10(moyenne de 10^(L/10)) — moyenne physique d'un niveau sonore
    db_px = bruit_db[r, c]
    okdb = np.isfinite(db_px)
    db_energ = 10 * np.log10(cnt(okdb, 10 ** (db_px / 10)) / cnt(okdb))
    part_fr = cnt(ok & ~np.isfinite(ch_cls[r, c])) / n_data

has = n_data > 0
classe_mode = np.where(has, par_classe.argmax(axis=1) + 1, 0)
classe_max = np.where(has, 7 - np.argmax(par_classe[:, ::-1] > 0, axis=1), 0)

stats = pd.DataFrame({
    "n_pix_perimetre": n_perim,
    "n_pix_donnee": n_data,
    "couverture": (n_data / n_theo).round(4),               # part du carreau avec une classe de bruit
    "classe_norm_moy": classe_norm_moy.round(4),            # 0 (calme) – 1 (classe 7)
    "classe_moy": classe_moy.round(3),
    "classe_mode": pd.array(classe_mode, dtype="Int8"),
    "classe_max": pd.array(classe_max, dtype="Int8"),
    "part_classe_ge4": parts[:, 3:].sum(axis=1).round(4),   # part ≥ 60 dB (zone « altérée » et pire)
    **{f"part_classe_{k}": parts[:, k - 1].round(4) for k in range(1, 8)},
    "bruit_db_energ": np.round(db_energ, 1),
    "part_pix_fr": part_fr.round(4),                        # part des pixels venant d'ORHANE
})
stats.loc[~has, ["classe_mode", "classe_max"]] = pd.NA

gdf_c = gpd.GeoDataFrame(pd.concat([carreaux.drop(columns="geometry"), stats], axis=1),
                         geometry=carreaux.geometry, crs=TARGET_CRS)
gdf_c = gdf_c[has & (stats["couverture"] >= CARREAU_MIN_COUVERTURE)].reset_index(drop=True)
centres = gdf_c.geometry.centroid
gdf_c["x_lv95"], gdf_c["y_lv95"] = centres.x.round(1), centres.y.round(1)

gdf_c.to_file(OUT_DIR / f"{CARREAU_STEM}.gpkg", layer="bruit_carreau200", driver="GPKG", engine="pyogrio")
gdf_c.to_parquet(OUT_DIR / f"{CARREAU_STEM}.parquet")
gdf_c.drop(columns="geometry").to_csv(OUT_DIR / f"{CARREAU_STEM}.csv", index=False)

print(f"{len(gdf_c):,} carreaux exportés "
      f"(dont {(gdf_c['couverture'] < 0.999).sum():,} partiellement couverts : bord, lac, frontière de données)")
for f in sorted(OUT_DIR.glob(f"{CARREAU_STEM}.*")):
    print(f"{f}  ({f.stat().st_size / 1e6:,.1f} Mo)")
gdf_c[[CARREAU_ID, "couverture", "classe_norm_moy", "classe_mode", "classe_max",
       "part_classe_ge4", "bruit_db_energ"]].describe(include="all")


200 m, 166,762 carreaux — 22,127,419 pixels du périmètre rattachés (0 hors carroyage)
48,608 carreaux exportés (dont 4,439 partiellement couverts : bord, lac, frontière de données)
sorties_bruit/bruit_classes_grand_geneve_carreau200.csv  (6.7 Mo)
sorties_bruit/bruit_classes_grand_geneve_carreau200.gpkg  (14.8 Mo)
sorties_bruit/bruit_classes_grand_geneve_carreau200.parquet  (2.5 Mo)


,GRID_ID,couverture,classe_norm_moy,classe_mode,classe_max,part_classe_ge4,bruit_db_energ
count,48608,48608.000000,48608.000000,48608.0,48608.0,48608.000000,48608.000000
unique,48608,NaN,NaN,<NA>,<NA>,NaN,NaN
top,JU-367,NaN,NaN,<NA>,<NA>,NaN,NaN
freq,1,NaN,NaN,<NA>,<NA>,NaN,NaN
mean,NaN,0.961100,0.088635,1.428777,2.173737,0.084274,47.859898
std,NaN,0.161633,0.171228,0.984684,1.834333,0.222374,9.449461
min,NaN,0.002500,0.000000,1.0,1.0,0.000000,0.200000
25%,NaN,1.000000,0.000000,1.0,1.0,0.000000,45.000000
50%,NaN,1.000000,0.000000,1.0,1.0,0.000000,45.000000
75%,NaN,1.000000,0.100000,1.0,3.0,0.000000,51.800000


## Sources à citer

- Périmètre : « Source : Système d'information du territoire à Genève (SITG),
  extrait en date du 31.03.2025 », avec mention du traitement (CGU SITG, art. 9).
- ORHANE : « Source ATMO Auvergne-Rhône-Alpes / ACOUCITE / CEREMA », licence
  ODbL (jeux de données dérivés publiés sous ODbL).
- Bruit Suisse : à compléter selon la licence de la couche utilisée.
- Méthode : reclassement du Lr suisse selon les seuils Lden ORHANE
  (approximation Lr ≈ Lden).